## Creating Gold Fact Tables Using Spark

Transforming silver fact tables into business-ready gold fact tables using PySpark:
* **gld_fact_order_items** - Order line items with calculated metrics

**Key Features**:
- Spark transformations for business metrics
- Partitioned by date_id for query performance
- Optimized for analytics and BI workloads

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

print(f"Using catalog: {catalog_name}")

In [0]:
# Build order items fact table with business metrics

df_order_items = spark.table(f"{catalog_name}.silver.slv_order_items")

# Apply transformations
df_fact_order_items = (df_order_items
    .withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast("int"))
    .withColumn("coupon_code", F.coalesce(F.col("coupon_code"), F.lit("No Coupon")))
    .select(
        F.col("date_id"),
        F.col("dt").alias("transaction_date"),
        F.col("order_ts").alias("transaction_ts"),
        F.col("order_id").alias("transaction_id"),
        F.col("item_seq").alias("seq_no"),
        F.col("customer_id"),
        F.col("product_id"),
        F.col("channel"),
        F.col("coupon_code"),
        F.col("unit_price_currency"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("discount_pct"),
        F.col("tax_amount")
    )
)

# Write to gold with date partitioning
(df_fact_order_items
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("date_id")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")
)

print(f"✓ gld_fact_order_items: {df_fact_order_items.count():,} rows")

In [0]:
# Summary of gold fact tables

print("\n" + "="*60)
print("Gold Fact Tables Summary")
print("="*60)

tables = ["gld_fact_order_items"]

for table in tables:
    count = spark.table(f"{catalog_name}.gold.{table}").count()
    print(f"{table:25} : {count:,} rows")

print("="*60)
print("\ ✓ All gold fact tables built successfully!")

In [0]:
# Data quality checks - duplicate validation

df_fact_check = spark.table(f"{catalog_name}.gold.gld_fact_order_items")
total_rows = df_fact_check.count()
unique_rows = df_fact_check.select("transaction_id", "seq_no").distinct().count()
duplicates = total_rows - unique_rows

print(f"\n gld_fact_order_items:")
print(f"Total rows: {total_rows:,}")
print(f"Unique transactions: {unique_rows:,}")
print(f"Duplicates: {duplicates}")

# Check partitioning
partitions = spark.sql(f"SHOW PARTITIONS {catalog_name}.gold.gld_fact_order_items").count()
print(f"Partitions (by date_id): {partitions}")

if duplicates == 0:
    print("\n Data Quality Check PASSED - No duplicates found!")
else:
    print("\n Data Quality Check FAILED - Duplicates detected!")